# **Capstone Project: A Bi-Objective Evolutionary Approach to Feature Selection for Customer Value Prediction in Fintech**

# *Data Collection & Target Engineering*

## MBAI 5600G: Applied Integrative Analytics Capstone Project

### Group 7: Brennan Mason & Mohammad Shah
---

## Environment Setup

In [ ]:
# Specify base path to local directory
BASE_PATH = "/content/drive/Shareddrives/MBAI Capstone S S26 Group 7/"

In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# Install dependencies
!pip install fredapi
!pip install numpy-financial

## Data Ingestion

In [ ]:
import numpy as np
import pandas as pd
import gc

# Define path
path = BASE_PATH + "p2p-customer-value-prediction/data/raw/raw_loans.csv"

# Define data types for select columns to circumvent mixed-type warnings
dtypes = {
    0: "str", 19: "str", 49: "str", 59: "str", 118: "str", 129: "str",
    130: "str", 131: "str", 134: "str", 135: "str", 136: "str", 139: "str",
    145: "str", 146: "str", 147: "str"
}

# Initialize empty list to store filtered chunks
expired_chunks = []

# Load data in chunks
for chunk in pd.read_csv(path, chunksize=100_000, dtype=dtypes, low_memory=True):
  # Filter chunk for expired/matured loans only
  expired_mask = chunk["loan_status"].isin(["Fully Paid", "Charged Off"])
  expired_chunk = chunk[expired_mask]

  # Append filtered chunk to list
  expired_chunks.append(expired_chunk)

# Concatenate filtered chunks into df
expired_loans = pd.concat(expired_chunks, ignore_index=True)

# Delete list of chunks to release memory
del expired_chunks
gc.collect()

# Implement stratified downsampling using a 20% sample stratified by loan grade
sample_frac = 0.2
sampled_loans = expired_loans.groupby("grade", group_keys=False).sample(frac=sample_frac, random_state=42)

# Reset index
sampled_loans.reset_index(drop=True, inplace=True)

# Delete full df to release memory
del expired_loans
gc.collect()

# Inspect head
sampled_loans.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,49763530,NaN,7000.0,7000.0,7000.0,36 months,7.89,219.00,A,A5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,1614412,NaN,5000.0,5000.0,5000.0,36 months,6.03,152.18,A,A1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,19435976,NaN,10000.0,10000.0,10000.0,36 months,7.12,309.32,A,A3,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,44756872,NaN,12000.0,12000.0,12000.0,36 months,5.93,364.69,A,A1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,108855699,NaN,6700.0,6700.0,6700.0,36 months,5.32,201.77,A,A1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Check size and type of downsampled dataset
sampled_loans.info(verbose=True, show_counts=True, memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 269062 entries, 0 to 269061
Data columns (total 151 columns):
 #    Column                                      Non-Null Count   Dtype  
---   ------                                      --------------   -----  
 0    id                                          269062 non-null  object 
 1    member_id                                   0 non-null       float64
 2    loan_amnt                                   269062 non-null  float64
 3    funded_amnt                                 269062 non-null  float64
 4    funded_amnt_inv                             269062 non-null  float64
 5    term                                        269062 non-null  object 
 6    int_rate                                    269062 non-null  float64
 7    installment                                 269062 non-null  float64
 8    grade                                       269062 non-null  object 
 9    sub_grade                                   269062 non-nu

## Target Engineering

In [ ]:
from google.colab import userdata
from fredapi import Fred

# Initialize FRED API
fred = Fred(api_key=userdata.get("FRED_API_KEY"))

# Pull historical US Federal Funds Effective Rate series as proxy for risk-free interbank overnight rate (CDI)
cdi_rates = pd.DataFrame(fred.get_series("FEDFUNDS"), columns=["cdi_rate"]).reset_index()

# Clean up df
cdi_rates.rename(columns={"index": "date"}, inplace=True)  # Rename date column
cdi_rates["date"] = pd.to_datetime(cdi_rates["date"])  # Convert date string to datetime type
cdi_rates["cdi_rate"] = cdi_rates["cdi_rate"] / 100  # Convert rate to decimal

# Engineer year column
cdi_rates["year"] = cdi_rates["date"].dt.year

# Compute annual average CDI rates
annual_avg_cdi_rates = cdi_rates.groupby("year")["cdi_rate"].mean().reset_index()
annual_avg_cdi_rates.rename(columns={"cdi_rate": "avg_cdi_rate"}, inplace=True)

# Inspect results
annual_avg_cdi_rates

,year,avg_cdi_rate
0,1954,0.010083
1,1955,0.017850
2,1956,0.027283
3,1957,0.031050
4,1958,0.015725
...,...,...
68,2022,0.016833
69,2023,0.050242
70,2024,0.051433
71,2025,0.042125


In [ ]:
# Convert relevant date columns to datetime type
sampled_loans["issue_d"] = pd.to_datetime(sampled_loans["issue_d"], format="%b-%Y")
sampled_loans["last_pymnt_d"] = pd.to_datetime(sampled_loans["last_pymnt_d"], format="%b-%Y")

# Engineer issue year column for merge
sampled_loans["issue_yr"] = sampled_loans["issue_d"].dt.year

# Pull annual average CDI rate based on issue year
sampled_loans = sampled_loans.merge(annual_avg_cdi_rates, how="left", left_on="issue_yr", right_on="year")
sampled_loans.drop(columns=["year"], inplace=True)

# Convert interest rate to decimal
sampled_loans["int_rate"] = sampled_loans["int_rate"] / 100

# Compute spread
sampled_loans["spread"] = ((sampled_loans["int_rate"] + 1) / (sampled_loans["avg_cdi_rate"] + 1)) - 1

# Engineer issue year-month column to define origination cohort (i.e., "vintage") and time period (t)
sampled_loans["issue_yr_mnth"] = sampled_loans["issue_d"].dt.to_period("M")

# Group by vintage and compute market parameters
vintage_market_params = sampled_loans.groupby("issue_yr_mnth")["spread"].agg(
    avg_spread="mean",  # Expected market return (Psi_m)
    min_spread="min"  # Risk-free return (Psi_f)
).reset_index()

# Ensure chronological ordering
vintage_market_params.sort_values(by="issue_yr_mnth", inplace=True)

# Compute smoothed market parameters with a 3-month rolling window to handle
# excessive volatility driven by small sample size in early vintages
vintage_market_params["avg_spread"] = vintage_market_params["avg_spread"].rolling(window=3, min_periods=1).mean()
vintage_market_params["min_spread"] = vintage_market_params["min_spread"].rolling(window=3, min_periods=1).min()

# Pull smoothed market parameters based on vintage
sampled_loans = sampled_loans.merge(vintage_market_params, how="left", on="issue_yr_mnth")

> **Notes**
> * Market parameters are estimated at the vintage level to prevent temporal leakage
> * Market parameters are smoothed over a 3-month rolling window to mitigate instability arising from small sample size in early LendingClub vintages

In [ ]:
# Compute lifetime realized ROI
sampled_loans["return"] = (sampled_loans["total_pymnt"] - sampled_loans["funded_amnt"]) / sampled_loans["funded_amnt"]

# Compute average returns by customer segment (i.e., loan grade) at each time period (Phi_ct)
segment_returns = sampled_loans.groupby(["grade", "issue_yr_mnth"])["return"].mean().reset_index()
segment_returns.rename(columns={"return": "segment_return"}, inplace=True)

# Compute average returns for entire market (i.e., customer base/portfolio) at each time period (Phi_mt)
market_returns = sampled_loans.groupby("issue_yr_mnth")["return"].mean().reset_index()
market_returns.rename(columns={"return": "market_return"}, inplace=True)

# Merge on time period
vintage_returns = pd.merge(segment_returns, market_returns, how="inner", on="issue_yr_mnth")

# Sort chronologically and inspect results
vintage_returns.sort_values(by=["issue_yr_mnth", "grade"]).reset_index(drop=True)

,grade,issue_yr_mnth,segment_return,market_return
0,A,2007-07,0.123006,0.139316
1,B,2007-07,0.155626,0.139316
2,A,2007-08,0.069197,-0.013930
3,B,2007-08,0.158284,-0.013930
4,C,2007-08,-0.245770,-0.013930
...,...,...,...,...
889,A,2018-12,0.010336,0.017171
890,B,2018-12,0.014551,0.017171
891,C,2018-12,0.017646,0.017171
892,D,2018-12,0.027857,0.017171


> **Notes**
> * The *LendingClub* dataset lacks longitudinal customer-level cash flows
> * Segment-level returns are used as a proxy to enable volatility estimation
>   * This customer segmentation is consistent with Buhl & Heinrich's approach
> * Segments are defined by loan grade due to its association with credit risk
> * Return is computed as lifetime realized ROI rather than contractual spread to assess actual outcomes

In [ ]:
# Define function to compute beta
def compute_beta(df):
  """
  Computes and returns beta value (i.e., systematic risk) given the series of segment returns and market returns in the provided DataFrame.

  Parameters:
  -----------
      - df (pandas.core.frame.DataFrame): DataFrame containing segment returns and market returns.

  Returns:
  --------
      - float: Computed beta value.
  """
  # Compute covariance between segment returns and market returns
  cov = df[["segment_return", "market_return"]].cov().iloc[0, 1]

  # Compute variance of market returns
  var = df["market_return"].var()

  # Compute and return beta
  return cov / var if var != 0 else 0

# Compute beta for each customer segment
segment_betas = vintage_returns.groupby("grade").apply(compute_beta, include_groups=False).reset_index(name="beta")

# Pull beta based on customer segment (i.e., loan grade)
sampled_loans = sampled_loans.merge(segment_betas, how="left", on="grade")

# Inspect results
segment_betas

,grade,beta
0,A,0.404783
1,B,0.765077
2,C,1.125921
3,D,1.490513
4,E,1.730945
5,F,1.916665
6,G,2.626782


> **Notes**
> * As expected, $\beta$ increases monotonically as credit rating (i.e., loan grade) degrades

In [ ]:
import numpy_financial as npf

# Compute discounting rate (i)
sampled_loans["disc_rate"] = sampled_loans["min_spread"] + sampled_loans["beta"] * (sampled_loans["avg_spread"] - sampled_loans["min_spread"])

# Convert annual discounting rate to monthly
sampled_loans["disc_rate"] = (1 + sampled_loans["disc_rate"]) ** (1 / 12) - 1

# Compute observed loan duration in months (T)
sampled_loans["loan_duration"] = (
    (sampled_loans["last_pymnt_d"].dt.year - sampled_loans["issue_d"].dt.year) * 12 +
    (sampled_loans["last_pymnt_d"].dt.month - sampled_loans["issue_d"].dt.month)
)

# Set lower bound for T at 1 to avoid division by zero errors
sampled_loans["loan_duration"] = sampled_loans["loan_duration"].clip(lower=1)

# Compute average monthly cash flow (delta)
sampled_loans["avg_monthly_cash_flow"] = sampled_loans["total_pymnt"] / sampled_loans["loan_duration"]

# Compute risk-adjusted revenue (RAR)
sampled_loans["rar"] = -npf.pv(rate=sampled_loans["disc_rate"], nper=sampled_loans["loan_duration"], pmt=sampled_loans["avg_monthly_cash_flow"], when="begin")

> **Notes**
> * A constant-payment annuity approximation is used due to lack of period-level cash flow observations
> * The annuity due formulation (i.e., `when="begin"`) is used as per Machado & Karray's RAR definition, which initializes the time counter $t$ at 0

## Target Inspection

In [ ]:
# Inspect statistical summary of results
print("RAR ($) Descriptive Statistics:")
print(sampled_loans["rar"].describe().round(2))

# Compare with statistical summary of funded amount
print("\nFunded Amount ($) Descriptive Statistics:")
print(sampled_loans["funded_amnt"].describe().round(2))

# Check statistical summary of discounting rate
print("\nDiscounting Rate Descriptive Statistics:")
print(sampled_loans["disc_rate"].describe().round(4))

RAR ($) Descriptive Statistics:
count    268615.00
mean      13352.55
std        9094.52
min          35.69
25%        6249.46
50%       10863.35
75%       18485.39
max       50915.73
Name: rar, dtype: float64

Funded Amount ($) Descriptive Statistics:
count    269062.00
mean      14435.33
std        8717.44
min        1000.00
25%        8000.00
50%       12000.00
75%       20000.00
max       40000.00
Name: funded_amnt, dtype: float64

Discounting Rate Descriptive Statistics:
count    269062.0000
mean          0.0102
std           0.0026
min           0.0023
25%           0.0084
50%           0.0100
75%           0.0119
max           0.0214
Name: disc_rate, dtype: float64


In [ ]:
# Specify columns of interest
target_cols = ["loan_status", "grade", "funded_amnt", "loan_duration", "disc_rate", "rar"]

# Inspect sample of fully paid grade A loans
mask = (sampled_loans["loan_status"] == "Fully Paid") & (sampled_loans["grade"] == "A")
print("Fully Paid Grade A Loans")
print(sampled_loans[mask][target_cols].sample(10, random_state=42))

# Inspect sample of fully paid grade G loans
mask = (sampled_loans["loan_status"] == "Fully Paid") & (sampled_loans["grade"] == "G")
print("\nFully Paid Grade G Loans")
print(sampled_loans[mask][target_cols].sample(10, random_state=42))

# Inspect sample of charged off loans
mask = sampled_loans["loan_status"] == "Charged Off"
print("\nCharged Off Loans")
print(sampled_loans[mask][target_cols].sample(10, random_state=42))

Fully Paid Grade A Loans
      loan_status grade  funded_amnt  loan_duration  disc_rate           rar
10249  Fully Paid     A       5000.0            6.0   0.006258   5056.230078
30104  Fully Paid     A      10000.0            7.0   0.005449  10236.857361
46893  Fully Paid     A      25000.0           21.0   0.006594  25801.293668
11630  Fully Paid     A      23000.0           36.0   0.006844  22710.722085
27174  Fully Paid     A      28000.0           31.0   0.006530  27536.923776
3673   Fully Paid     A      18000.0           15.0   0.006499  18378.368278
46720  Fully Paid     A      20000.0           31.0   0.006181  21236.992228
27625  Fully Paid     A      21000.0           36.0   0.007415  20761.857403
33781  Fully Paid     A      35000.0           26.0   0.006184  36109.397194
24371  Fully Paid     A      10000.0           36.0   0.006923  10043.665378

Fully Paid Grade G Loans
       loan_status grade  funded_amnt  loan_duration  disc_rate           rar
267368  Fully Paid     G

In [ ]:
import plotly.graph_objects as go

# Visualize RAR distribution
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=sampled_loans["rar"],
    nbinsx=40,
    marker=dict(color="steelblue"),
    name="RAR"
))

fig.update_layout(
    title="Distribution of Risk-Adjusted Revenue",
    xaxis_title="Risk-Adjusted Revenue ($)",
    yaxis_title="Frequency",
    template="seaborn",
    font=dict(
        family="Helvetica Neue, sans-serif",
        color="#666666"
    ),
    width=1000,
    height=600
)

fig.update_yaxes(ticklabelstandoff=5)
fig.update_xaxes(ticklabelstandoff=5)

fig.show()

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# Compute average RAR by loan grade
rar_by_grade = sampled_loans.groupby("grade")["rar"].mean()

# Visualize average RAR by loan grade
fig = go.Figure()

fig.add_trace(go.Bar(
    x=rar_by_grade.index,
    y=rar_by_grade.values,
    marker=dict(color="steelblue"),
    name="RAR"
))

fig.update_layout(
    title="Average Risk-Adjusted Revenue by Loan Grade",
    xaxis_title="Loan Grade",
    yaxis_title = "Average Risk-Adjusted Revenue ($)",
    template="seaborn",
    font=dict(
        family="Helvetica Neue, sans-serif",
        color="#666666"
    ),
    width=1000,
    height=600
)

fig.update_traces(
    texttemplate="$%{y:,.2f}",
    textposition="inside",
    insidetextanchor="end",
    textfont=dict(
        family="Helvetica Neue, sans-serif",
        color="white"
    )
)

fig.update_yaxes(ticklabelstandoff=5)
fig.update_xaxes(ticklabelstandoff=5)

fig.show()

> **Notes**
> * As expected, higher levels of credit risk (as proxied by loan grade) are generally associated with higher RAR
>   * This aligns with Customer Portfolio Theory, which proposes a positive correlation between customer risk and expected return

In [ ]:
# Visualize beta by loan grade
fig = go.Figure()

fig.add_trace(go.Bar(
    x=segment_betas["grade"],
    y=segment_betas["beta"],
    marker=dict(color="steelblue"),
    name="Beta"
))

fig.update_layout(
    title="Systematic Risk (𝛽) by Loan Grade",
    xaxis_title="Loan Grade",
    yaxis_title="Systematic Risk (𝛽)",
    template="seaborn",
    font=dict(
        family="Helvetica Neue, sans-serif",
        color="#666666"
    ),
    width=1000,
    height=600
)

fig.update_traces(
    texttemplate="%{y:.2f}",
    textposition="inside",
    insidetextanchor="end",
    textfont=dict(
        family="Helvetica Neue, sans-serif",
        color="white"
    )
)

fig.update_yaxes(ticklabelstandoff=5)
fig.update_xaxes(ticklabelstandoff=5)

fig.show()

In [ ]:
# Compute average RAR by loan status
rar_by_status = sampled_loans.groupby("loan_status")["rar"].mean()

# Visualize average RAR by loan status
fig = go.Figure()

fig.add_trace(go.Bar(
    y=rar_by_status.index,
    x=rar_by_status.values,
    orientation="h",
    marker=dict(color="steelblue"),
    name="RAR"
))

fig.update_layout(
    title="Average Risk-Adjusted Revenue by Loan Status",
    xaxis_title="Average Risk-Adjusted Revenue ($)",
    yaxis_title = "",
    template="seaborn",
    font=dict(
        family="Helvetica Neue, sans-serif",
        color="#666666"
    ),
    width=1000,
    height=600
)

fig.update_traces(
    texttemplate="$%{x:,.2f} ",
    textposition="inside",
    insidetextanchor="end",
    textfont=dict(
        family="Helvetica Neue, sans-serif",
        color="white"
    )
)

fig.update_yaxes(ticklabelstandoff=5)
fig.update_xaxes(ticklabelstandoff=5)

fig.show()

In [ ]:
# Compute loan count by loan status
loans_by_status = sampled_loans.groupby("loan_status")["loan_status"].count()

# Visualize loan status distribution
fig = go.Figure()

fig.add_trace(go.Bar(
    y=loans_by_status.index,
    x=loans_by_status.values,
    orientation="h",
    marker=dict(color="steelblue"),
    name="Loan Count"
))

fig.update_layout(
    title="Distribution of Loan Statuses",
    xaxis_title="Frequency",
    yaxis_title = "",
    template="seaborn",
    font=dict(
        family="Helvetica Neue, sans-serif",
        color="#666666"
    ),
    width=1000,
    height=600
)

fig.update_traces(
    texttemplate="%{x:,.0f} ",
    textposition="inside",
    insidetextanchor="end",
    textfont=dict(
        family="Helvetica Neue, sans-serif",
        color="white"
    )
)

fig.update_yaxes(ticklabelstandoff=5)
fig.update_xaxes(ticklabelstandoff=5)

fig.show()

In [ ]:
sampled_loans["loan_status"].value_counts(normalize=True)

,proportion
loan_status,
Fully Paid,0.799998
Charged Off,0.200002


In [ ]:
# Compute default rate by loan grade
default_mask = sampled_loans["loan_status"] == "Charged Off"
defaults_by_grade = sampled_loans[default_mask].groupby("grade")["loan_status"].count()
defaults_by_grade.rename("n_defaults", inplace=True)
loans_by_grade = sampled_loans.groupby("grade")["loan_status"].count()
loans_by_grade.rename("n_loans", inplace=True)
rate_by_grade = pd.concat([defaults_by_grade, loans_by_grade], axis=1)
rate_by_grade["default_rate"] = rate_by_grade["n_defaults"] / rate_by_grade["n_loans"]

# Visualize default rate by loan grade
fig = go.Figure()

fig.add_trace(go.Bar(
    x=rate_by_grade.index,
    y=rate_by_grade["default_rate"],
    marker=dict(color="steelblue"),
    name="Default Rate"
))

fig.update_layout(
    title="Default Rate by Loan Grade",
    xaxis_title="Loan Grade",
    yaxis_title="Default Rate",
    template="seaborn",
    font=dict(
        family="Helvetica Neue, sans-serif",
        color="#666666"
    ),
    width=1000,
    height=600
)

fig.update_traces(
    texttemplate="%{y:.1%}",
    textposition="inside",
    insidetextanchor="end",
    textfont=dict(
        family="Helvetica Neue, sans-serif",
        color="white"
    )
)

fig.update_yaxes(ticklabelstandoff=5, tickformat=".0%")
fig.update_xaxes(ticklabelstandoff=5)

fig.show()

## Data Export

In [ ]:
# Drop rows where RAR could not be computed
sampled_loans.dropna(subset=["rar"], inplace=True)
sampled_loans.reset_index(drop=True, inplace=True)

In [ ]:
# Define path
path = BASE_PATH + "p2p-customer-value-prediction/data/interim/sampled_loans.csv"

# Write downsampled dataset with target to CSV
sampled_loans.to_csv(path, index=False)

---
## References
1. M. R. Machado and S. Karray, “Applying hybrid machine learning algorithms to assess customer risk-adjusted revenue in the financial industry,” Electron. Commer. Res. Appl., vol. 56, p. 101202, Nov. 2022, doi: 10.1016/j.elerap.2022.101202.

2. R. Dhar and R. Glazer, “Hedging Customers.,” Harv. Bus. Rev., vol. 81, no. 5, pp. 86–92, May 2003.

3. H. U. Buhl and B. Heinrich, “Valuing Customer Portfolios under Risk-Return-Aspects: A Model-based Approach and its Application in the Financial Services Industry,” Acad. Mark. Sci. Rev., vol. 12, no. 5, 2008, doi: 10.5283/epub.23202.